# Yelp Review Sentiment Analysis  
### CS3 Case Study Example

In this notebook, we explore how customer review text can be used to predict whether a restaurant receives a high or low rating.

Our goal is to walk through a simplified machine learning workflow and understand how text data can be transformed into meaningful insights.

In [15]:
# ======================================
# Yelp Sentiment Model - CS3 Case Study
# ======================================

import pandas as pd
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

## 1. Load and Inspect the Data

We begin by loading a dataset of Yelp restaurant reviews. Each review includes a star rating and written feedback from a customer.

Take a moment to explore the structure of the dataset.

In [16]:
# ============================================================
# 1. Load Data
# ============================================================

df = pd.read_csv("Yelp Restaurant Reviews.csv")

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df = df.rename(columns={
    "yelp_url": "url",
    "review_text": "text"
})

df.head()

,url,rating,date,text
0,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,1/22/2022,All I can say is they have very good ice cream...
1,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,6/26/2022,Nice little local place for ice cream.My favor...
2,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,8/7/2021,A delicious treat on a hot day! Staff was very...
3,https://www.yelp.com/biz/sidney-dairy-barn-sidney,4,7/28/2016,This was great service and a fun crew! I got t...
4,https://www.yelp.com/biz/sidney-dairy-barn-sidney,5,6/23/2015,This is one of my favorite places to get ice c...


## 2. Create a Sentiment Label

To simplify the problem, we convert star ratings into a binary outcome:
- Positive reviews (4–5 stars)
- Negative reviews (1–3 stars)

In [17]:
# ============================================================
# 2. Create Sentiment Label
# ============================================================
# TODO: Why might we group ratings this way, specifically classifying a 3 as negative?

df["sentiment"] = df["rating"].apply(
    lambda x: "Positive" if x >= 4 else "Negative"
)

df["sentiment"].value_counts()

,count
sentiment,
Positive,9932
Negative,2983


## 3. Clean the Review Text

Before analyzing text, we need to clean it by:
- Converting to lowercase  
- Removing punctuation  
- Standardizing spacing  

This helps ensure that words are treated consistently.

In [18]:
# ============================================================
# 3. Clean Text
# ============================================================

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["text_clean"] = df["text"].apply(clean_text)

df[["text", "text_clean"]].head()

,text,text_clean
0,All I can say is they have very good ice cream...,all i can say is they have very good ice cream...
1,Nice little local place for ice cream.My favor...,nice little local place for ice cream my favor...
2,A delicious treat on a hot day! Staff was very...,a delicious treat on a hot day staff was very ...
3,This was great service and a fun crew! I got t...,this was great service and a fun crew i got th...
4,This is one of my favorite places to get ice c...,this is one of my favorite places to get ice c...


## 4. Split the Data

We divide the dataset into training and testing sets.

The training data is used to build the model, while the test data helps us evaluate how well it performs on new, unseen reviews.

In [19]:
# ============================================================
# 4. Train-Test Split
# ============================================================

X = df["text_clean"]
y = df["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training size:", len(X_train))
print("Test size:", len(X_test))

Training size: 10332
Test size: 2583


## 5. Convert Text into Features

Machine learning models cannot directly interpret text, so we convert words into numerical features using TF-IDF.

TF-IDF helps capture which words are most important in each review.

In [20]:
# ============================================================
# 5. Convert Text to Features
# ============================================================
# We use TF-IDF to turn words into numerical features.
# TODO: Why might this be useful?

tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=100
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

## 6. Train a Classification Model

We now train a logistic regression model to predict whether a review is positive or negative based on its text.

This model will learn patterns in the language used by customers.

In [21]:
# ============================================================
# 6. Train Model
# ============================================================
# TODO: Try adjusting parameters and observe results

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

predictions = model.predict(X_test_tfidf)

## 7. Evaluate Model Performance

After training the model, we evaluate how well it performs on the test data.

Consider:
- How accurate is the model?
- Does it perform better on positive or negative reviews?

## 8. Interpreting the Model

Finally, we can examine which words are most strongly associated with positive or negative reviews.

This helps us answer the central question:
What language patterns are most predictive of high ratings?

In [22]:
# ============================================================
# 7. Evaluate Model
# ============================================================

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

Accuracy: 0.826945412311266

Classification Report:
              precision    recall  f1-score   support

    Negative       0.68      0.47      0.56       597
    Positive       0.85      0.94      0.89      1986

    accuracy                           0.83      2583
   macro avg       0.77      0.70      0.72      2583
weighted avg       0.81      0.83      0.81      2583



## Reflection Questions

- Why might some common words not be useful for predicting sentiment?
- What types of words seem most important for distinguishing positive vs. negative reviews?
- How might you improve this model?

These questions will guide your analysis and help you connect the results to real-world insights.